# Génération de masques de neige
<br/>
Dans ce notebook, nous allons générer des masques de neige sur des images de la tuile 31TCH.<br/>
Pour cela, nous allons filtrer le load_collection avec deux propriétés (nom de la tuile et couverture nuageuse), calculer le NDSI (indice de neige), appliquer un seuil puis générer un masque binaire en sortie.<br/>
<br/>
Import et installation des librairies nécessaires

In [ ]:
try:
    import openeo
except:
    !pip install openeo
    import openeo

In [ ]:
# Connexion à l'aggrégateur
con = openeo.connect("https://aggregator.geodes-lab.fr")
con.authenticate_oidc() 

## Création du datacube et du process graph

**Etape 1 :** Utiliser le process "load_collection" pour charger les données.

In [ ]:
df = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=["2022-01-01","2022-08-01"],
    bands=["B3", "B11"],
    max_cloud_cover=10,
    properties=[openeo.collection_property("tilename") == "T31TCH"]    
)
df

**Etape 2 :** Calcul du NDSI avec un reducer sur les bandes.

In [ ]:
green = df.band("B3")
swir = df.band("B11")

df_ndsi = (green - swir) / (green + swir)
df_ndsi

**Etape 3 :** Appliquer un seuil pour détecter la neige

In [ ]:
df_ndsi_with_threshold = df_ndsi > 0.4
df_ndsi_with_threshold

**Etape 4 :** On ajoute le save_result avec le profil voulu.

In [ ]:
df_result = df_ndsi_with_threshold.save_result(
    format="COG",
    options={"dtype": "uint8", "nodata": 255}
)
df_result

## Création et lancement du job

In [ ]:
# On veut du Zarr en sortie
job_df = df_result.create_job(title="Masques de neige 31TCH")

In [ ]:
job_df.start_and_wait()  # environ 6 min

**Une fois que le job est terminé (status=finished), on récupère les résultats**

In [ ]:
# Aperçu des résultats
results = job_df.get_results()
results

In [ ]:
# Téléchargement
results.download_files("./results_snow/")

## Visualisation des résultats

In [ ]:
import rasterio
import matplotlib.pyplot as plt

In [ ]:
file_path = "openeo_2022-01-17T000000_cog.tif"

with rasterio.open(f"results_snow/{file_path}") as src:
    data = src.read(1, masked=True)

# 0 = black, 1 = white, NoData = transparent
cmap = plt.colormaps["gray"].copy()
cmap.set_bad(color="none")

plt.figure()
plt.imshow(data, cmap=cmap, vmin=0, vmax=1)
plt.axis("off")
plt.show()

In [ ]:
# Clean

import shutil

shutil.rmtree("./results_snow/")